# Solving the Symmetric Positive Definite System

We consider the linear system

$$
Aw=b,
$$

where

$$
A\in\mathbb R^{n\times n}
$$

is a symmetric positive-definite matrix.

The objective of this notebook is to solve the system using the most appropriate algorithm depending on the size and structure of the problem.

The notebook implements

1. Cholesky factorization
2. Conjugate Gradient (CG)
3. Automatic solver selection


# Computational Complexity

For dense matrices:

| Method | Complexity |
|----------|-----------|
| Gaussian Elimination | $O(n^3)$ |
| LU Decomposition | $\frac23 n^3$ |
| Cholesky | $\frac13 n^3$ |

Thus Cholesky requires approximately half the work of LU decomposition.

For very large systems iterative methods become attractive.

The most important iterative method for SPD matrices is the Conjugate Gradient method.

In [1]:
import numpy as np
import time

In [2]:

def load_csv(A_file, b_file):
    import pandas as pd

    A = pd.read_csv(A_file, header=None).values
    b = pd.read_csv(b_file, header=None).values.flatten()

    return A.astype(float), b.astype(float)


# SPD Verification

Before solving we verify

$$
A=A^T
$$

and attempt a Cholesky factorization.

Successful factorization implies positive definiteness.

In [3]:
def is_symmetric(A, tol=1e-12):
    return np.allclose(A, A.T, atol=tol)


def is_positive_definite(A):
    try:
        np.linalg.cholesky(A)
        return True
    except np.linalg.LinAlgError:
        return False

In [4]:
def forward_substitution(L, b):

    n = len(b)

    y = np.zeros(n)

    for i in range(n):

        y[i] = ( b[i] - np.dot(L[i,:i], y[:i])) / L[i,i]

    return y


def backward_substitution(U, y):

    n = len(y)

    x = np.zeros(n)

    for i in range(n-1, -1, -1):

        x[i] = ( y[i] - np.dot(U[i,i+1:], x[i+1:])) / U[i,i]

    return x

# Mathematical Background

A matrix $A$ is called symmetric positive definite (SPD) if

$$
A=A^T
$$

and

$$
x^TAx>0
\qquad
\forall x\neq0.
$$

Important properties:

- all eigenvalues are positive,
- $A$ is nonsingular,
- Cholesky factorization exists and is unique.

Therefore there exists a unique lower triangular matrix $L$ such that

$$
A=LL^T.
$$

The system

$$
Aw=b
$$

becomes

$$
LL^Tw=b.
$$

Let

$$
y=L^{-1}b.
$$

Then

$$
Ly=b,
$$

followed by

$$
L^Tw=y.
$$

This reduces the solution of the original problem to two triangular systems.

# Cholesky Solver

The Cholesky factorization is

$$
A=LL^T.
$$

The solution is obtained from

$$
Ly=b
$$

and

$$
L^Tw=y.
$$

In [5]:
def cholesky_solver(A, b):

    L = np.linalg.cholesky(A)

    y = forward_substitution(L, b)

    w = backward_substitution(L.T, y)

    return w

# Conjugate Gradient Solver

For large systems it may be preferable to avoid matrix factorization.

## Method

Consider

$$
Aw=b.
$$

Define

$$
r_k=b-Aw_k.
$$

The residual vector measures the current error.

The Conjugate Gradient algorithm constructs search directions

$$
p_k
$$

that satisfy

$$
p_i^TAp_j=0,
\qquad i\neq j.
$$

These directions are called $A$-conjugate.

The iterations are

$$
\alpha_k=
\frac{r_k^Tr_k}
     {p_k^TAp_k}
$$

$$
w_{k+1}
=
w_k+\alpha_kp_k
$$

$$
r_{k+1}
=
r_k-\alpha_kAp_k
$$

$$
\beta_k
=
\frac{r_{k+1}^Tr_{k+1}}
     {r_k^Tr_k}
$$

$$
p_{k+1}
=
r_{k+1}
+
\beta_kp_k.
$$

For exact arithmetic, convergence occurs in at most $n$ iterations.

In practice convergence is usually much faster.

In [ ]:
def conjugate_gradient( A, b, tol=1e-10, maxiter=None):

    n = len(b)

    if maxiter is None:
        maxiter = n

    x = np.zeros(n)

    r = b - A @ x

    p = r.copy()

    rsold = np.dot(r, r)

    for k in range(maxiter):

        Ap = A @ p

        alpha = rsold / np.dot(p, Ap)

        x = x + alpha * p

        r = r - alpha * Ap

        rsnew = np.dot(r, r)

        if np.sqrt(rsnew) < tol:

            return x, k + 1

        beta = rsnew / rsold

        p = r + beta * p

        rsold = rsnew

    return x, maxiter

# Automatic Solver Selection

The following strategy is used.

If

$$
n < 3000
$$

use Cholesky.

Otherwise use Conjugate Gradient.

The threshold may be modified depending on available memory.

In [ ]:
def solve_spd(A, b, threshold=3000, cg_tol=1e-10):

    n = A.shape[0]

    if not is_symmetric(A):
        raise ValueError("Matrix is not symmetric")

    if not is_positive_definite(A):
        raise ValueError("Matrix is not SPD")

    start = time.perf_counter()

    if n < threshold:

        method = "Cholesky"

        x = cholesky_solver(A, b)

        iterations = None

    else:

        method = "Conjugate Gradient"

        x, iterations = conjugate_gradient( A, b, tol=cg_tol)

    elapsed = time.perf_counter() - start


    # try:
    #     cond = np.linalg.cond(A)
    # except:
    #     cond = np.nan

    report = {
        "method": method,
        "n": n,
        "iterations": iterations,
        "time_seconds": elapsed
    }

    return x, report

In [8]:
A = np.array([
    [4,1,1],
    [1,3,0],
    [1,0,2]
])

b = np.array([1,2,3])

w, report = solve_spd(A,b)

print("\nSolution:")
print(w)

print("\nDiagnostics:")
for k,v in report.items():
    print(f"{k}: {v}")


Solution:
[-0.36842105  0.78947368  1.68421053]

Diagnostics:
method: Cholesky
n: 3
condition_number: 3.3240331760052277
iterations: None
residual_norm: 0.0
relative_residual: 0.0
time_seconds: 0.00017504999414086342
